In [ ]:
import json
import os
import sys

import tensorflow as tf
import yaml

sys.path.append(os.path.abspath("../"))
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import glob

import matplotlib.pyplot as plt
import numpy as np
import sklearn

from train.models import FullModel
from util import dataset as u_dataset
from util import dataset_io as u_dataset_io
from util import metrics as u_metrics

dataset_utils = u_dataset.DatasetUtils(u_dataset.DatasetConfig())

In [ ]:
batch_size = 1840

model_name = "20260115-150838.keras"
path_to_models = "../../models/finished/"

In [ ]:
# Get encoder config
def load_config(config_path):
    with open(config_path) as f:
        config = yaml.safe_load(f)
    return config


config_dir = f"../../logs/fit/{model_name.split('.')[0]}/"
config = load_config(config_dir + "config.yaml")

encoder_architecture = config["model"]["encoder"]["architecture"]
classifier_architecture = config["model"]["classifier"]["architecture"]
categories_config = config["categories"]

#### Load Test Data


In [ ]:
path_to_test_data = glob.glob("../../data/tfrecords/test_ds_v3*.tfrecords")

test_ds = u_dataset_io.get_dataset(path_to_test_data)
# test_ds = test_ds.take(20)
test_ds = test_ds.batch(batch_size, drop_remainder=False)

#### Load Model


In [ ]:
model = FullModel.load(
    encoder_architecture,
    classifier_architecture,
    input_dims=config["model"]["encoder"]["input_dims"],
    filepath=path_to_models,
    filename=model_name,
    n_context=config["model"]["encoder"]["n_context"],
    only_train_encoder=config["model"]["encoder"]["only_train_encoder"],
    classifier_offsets=config["model"]["classifier"]["with_offsets"],
    encoder_only=False,
    verbose=True,
    n_meta=config["model"]["classifier"]["n_meta"],
    encoder_use_batch_norm=config["model"]["encoder"]["use_batch_norm"],
    classifier_use_batch_norm=config["model"]["classifier"]["use_batch_norm"],
    categories_config=config["categories"],
)
model.compile(optimizer=tf.keras.optimizers.Adam(), jit_compile=False)


#### Predict Data


In [ ]:
groundtruth_dataset = []

for x in test_ds:
    groundtruth_dataset.append(x)


# Take only image, camera, intrinsics for input
input_data = test_ds.map(lambda x: (x["image"], x["camera"], x["intrinsics"]))

# Convert tf.Data.Dataset into list
input_list = []
for x, y, z in input_data:
    input_list.append((x, y, z))

model.run_eagerly = True

predictions = [model.predict(x=batch) for batch in input_list]
metrics_list = [model.evaluate(x=batch, return_dict=True) for batch in groundtruth_dataset]

In [ ]:
# tf.print(tf.shape(input_list[0][0]))

# current_image_yuyv = input_list[0][0][19]

# image = u_image.convert_yuyv_to_rgb(current_image_yuyv)

# plt.imshow(image[...], cmap="gray")
# plt.title("Raw Example Image in Grayscale")
# plt.show()

#### Evaluate Models


In [ ]:
# Calculate mean for metrics over each batch
keys = metrics_list[0].keys()
stacked_values = {key: tf.stack([metrics[key] for metrics in metrics_list]) for key in keys}
metrics_mean = {key: float(tf.reduce_mean(stacked_values[key])) for key in keys}

# Print Metrics
for key, value in metrics_mean.items():
    print(f"{key}: {value:.4f}")

### Accuracy, Precision, Recall


In [ ]:
def reshape_tensors(object_tensors, reshape=True):
    # object_tensors = [tensor[object_name] for tensor in tensors]

    stacked_tensors = {
        key: tf.stack([object_tensor[key] for object_tensor in object_tensors])
        for key in object_tensors[0]
    }
    if reshape:
        return {
            key: tf.reshape(tensor, (-1, *tensor.shape[2:]))
            for key, tensor in stacked_tensors.items()
        }
    else:
        return stacked_tensors

In [ ]:
object_name = u_dataset.CategoryNames.INTERSECTIONS.value

groundtruth_reshaped = reshape_tensors([batch[object_name] for batch in groundtruth_dataset], True)
intersections_predictions_reshaped = reshape_tensors(
    [batch["results"][object_name] for batch in predictions], True
)
thresholds = np.linspace(0, 1, 40, endpoint=True)

conf_values_cla = [
    u_metrics.calculate_multiclass_metrics(
        intersections_predictions_reshaped, groundtruth_reshaped, cla, 0.1
    )
    for cla in thresholds
]


# conf_values_enc = [
#     get_conf_matrix(ball_predictions_reshaped, groundtruth_reshaped, enc, 0) for enc in thresholds
# ]

In [ ]:
encoder_threshold = 0.1
classifier_threshold = 0.9

In [ ]:
precision_cla = [x["precision_pooled"] for x in conf_values_cla]
recall_cla = [x["recall_pooled"] for x in conf_values_cla]

plt.plot(thresholds, precision_cla, label="Precision")
plt.plot(thresholds, recall_cla, label="Recall")
plt.title("Precision/Recall Curve for Classifier Threshold")
plt.axvline(classifier_threshold, ls="--", color="r")
plt.xlabel("Threshold")
plt.ylabel("Precision/Recall")
plt.legend()
plt.show()

In [ ]:
metrics = u_metrics.calculate_multiclass_metrics(
    intersections_predictions_reshaped,
    groundtruth_reshaped,
    classifier_threshold,
    encoder_threshold,
)

In [ ]:
confusion_matrix = metrics["confusion_matrix"]
confusion_matrix_pooled = metrics["confusion_matrix_pooled"]

tf.print("Precision: ", metrics["precision_pooled"])
tf.print("Recall: ", metrics["recall_pooled"])
tf.print("fp rate:", metrics["fp_rate"])
tf.print("fn rate: ", metrics["fn_rate"])

sklearn.metrics.ConfusionMatrixDisplay(
    confusion_matrix, display_labels=np.array(["None", "L", "T", "X"])
).plot()
sklearn.metrics.ConfusionMatrixDisplay(
    confusion_matrix_pooled, display_labels=np.array(["True", "False"])
).plot()

#### Write Metrics into a log file


In [ ]:
metrics_log = {
    "metrics": metrics_mean,
    "evaluation": {
        "encoder_threshold": encoder_threshold,
        "classifier_threshold": classifier_threshold,
        "precision": float(metrics["precision_pooled"]),
        "recall": float(metrics["recall_pooled"]),
        "confusion_matrix_pooled": (metrics["confusion_matrix_pooled"]).tolist(),
        "confusion_matrix": (metrics["confusion_matrix"]).tolist(),
        "fp_rate": float(
            confusion_matrix[0][1] / (confusion_matrix[0][1] + confusion_matrix[1][1])
        ),
        "fn_rate": float(
            confusion_matrix[1][0] / (confusion_matrix[1][0] + confusion_matrix[0][0])
        ),
    },
    # "fp_indices": a["fp_indices"].tolist(),
    # "fn_indices": a["fn_indices"].tolist(),
}

log_file = config_dir + "metrics.yaml"
with open(log_file, "w") as f:
    # dump config file into log
    yaml.dump(metrics_log, f, sort_keys=False, default_flow_style=False)

print(f"Configuration logged to {log_file}")

### Write predictions into file

In [ ]:
def batch_nms(
    boxes, scores, max_output_size_per_class=6, iou_threshold=0.5, score_threshold=float("-inf")
):
    """
    Apply NMS to each batch element.

    Args:
        boxes: Tensor of shape (B, N, 4)
        scores: Tensor of shape (B, N)
        max_output_size_per_class: Maximum number of boxes to keep per batch element
        iou_threshold: IoU threshold for NMS
        score_threshold: Minimum score threshold for keeping a box

    Returns:
        selected_indices: List of tensors, each containing the indices of selected boxes for a batch element
    """
    selected_indices = []
    for i in range(boxes.shape[0]):
        # Extract boxes and scores for the i-th batch element
        boxes_i = boxes[i]
        scores_i = scores[i]

        # Apply NMS
        selected = tf.image.non_max_suppression(
            boxes_i,
            scores_i,
            max_output_size_per_class,
            iou_threshold=iou_threshold,
            score_threshold=score_threshold,
        )
        selected_indices.append(selected)
    return selected_indices

In [ ]:
selected_indices = batch_nms(
    predictions[0]["results"]["intersections"]["boxes"],
    tf.reduce_max(predictions[0]["results"]["intersections"]["classification"], axis=-1),
)

tf.print(len(selected_indices))

In [ ]:
def coords_tensor_to_dict_list(tensor):
    return [
        {"x": float(x), "y": float(y), "confidence": float(conf)} for x, y, conf in tensor.numpy()
    ]

In [ ]:
best_logits = tf.gather(
    predictions[0]["results"]["intersections"]["logits"],
    predictions[0]["results"]["intersections"]["patch_indices"],
    batch_dims=1,
)  # (B, N)
classifier_preds = tf.reduce_max(
    predictions[0]["results"]["intersections"]["classification"], axis=-1
)  # (B, N)
positions = predictions[0]["results"]["intersections"]["positions"]  # (B, N, 2)

intersection_tresholding_mask = u_metrics.get_thresholding_mask(
    classifier_preds,
    classifier_threshold,
    best_logits,
    encoder_threshold,
)  # (B, N)

intersections_thresholded = tf.reshape(
    tf.where(
        tf.reshape(intersection_tresholding_mask, [-1]),
        tf.reshape(
            tf.argmax(predictions[0]["results"]["intersections"]["classification"], axis=-1), [-1]
        ),
        0,
    ),
    tf.shape(intersection_tresholding_mask),
)  # (B, N)

preds = []
for idx, name in enumerate(groundtruth_dataset[0]["name"]):
    frame_time = groundtruth_dataset[0]["frame_time"][idx]

    intersections_thresholded_supressed = tf.gather(intersections_thresholded[idx], selected_indices[idx])
    positions_supressed = tf.gather(positions[idx], selected_indices[idx])
    classifier_preds_supressed = tf.gather(classifier_preds[idx], selected_indices[idx])
    
    l_intersections = intersections_thresholded_supressed == 1
    l_positions = tf.boolean_mask(positions_supressed, l_intersections)
    l_confidence = tf.boolean_mask(classifier_preds_supressed, l_intersections)
    l_tensor = tf.concat([l_positions, tf.expand_dims(l_confidence, axis=-1)], axis=-1)

    t_intersections = intersections_thresholded_supressed == 2
    t_positions = tf.boolean_mask(positions_supressed, t_intersections)
    t_confidence = tf.boolean_mask(classifier_preds_supressed, t_intersections)
    t_tensor = tf.concat([t_positions, tf.expand_dims(t_confidence, axis=-1)], axis=-1)

    x_intersections = intersections_thresholded_supressed == 3
    x_positions = tf.boolean_mask(positions_supressed, x_intersections)
    x_confidence = tf.boolean_mask(classifier_preds_supressed, x_intersections)
    x_tensor = tf.concat([x_positions, tf.expand_dims(x_confidence, axis=-1)], axis=-1)

    sample = {
        "name": name.numpy().decode("utf-8"),
        "frame_time": int(frame_time.numpy()),
        "intersections": {
            "L": coords_tensor_to_dict_list(l_tensor),
            "T": coords_tensor_to_dict_list(t_tensor),
            "X": coords_tensor_to_dict_list(x_tensor),
        },
    }
    preds.append(sample)

with open("../../data/selected/" + "test_model" + ".json", "w") as f:
    json.dump(preds, f, indent=4)